Montage et chargement

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

base = '/content/drive/MyDrive/EnergyMind_TSYP14/data/'
df = pd.read_csv(base + 'quartier_unifie.csv', parse_dates=['tstp'])
df = df.sort_values(['LCLid', 'tstp']).reset_index(drop=True)

print(df.shape)
print(df.dtypes)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(952553, 9)
LCLid                        object
tstp                 datetime64[ns]
energy(kWh/hh)              float64
power_kW                    float64
Voltage                     float64
Global_intensity            float64
Temperature_synth           float64
Humidity_synth              float64
etait_manquant                 bool
dtype: object


Poids de fragilité par foyer

In [ ]:
rng = np.random.default_rng(123)

foyers = df['LCLid'].unique()
n_foyers = len(foyers)

poids_fragilite = rng.lognormal(mean=0, sigma=0.6, size=n_foyers)
poids_fragilite = poids_fragilite / poids_fragilite.mean()
poids_par_foyer = dict(zip(foyers, poids_fragilite))

print(pd.Series(poids_fragilite).describe())
print("Foyer le plus fragile :", max(poids_par_foyer, key=poids_par_foyer.get), "-", round(max(poids_par_foyer.values()), 2))
print("Foyer le plus robuste :", min(poids_par_foyer, key=poids_par_foyer.get), "-", round(min(poids_par_foyer.values()), 2))

count    41.000000
mean      1.000000
std       0.507451
min       0.204390
25%       0.618596
50%       0.863416
75%       1.394800
max       2.211682
dtype: float64
Foyer le plus fragile : MAC004247 - 2.21
Foyer le plus robuste : MAC003646 - 0.2


Budgets cibles et structures de suivi

In [ ]:
n_valid = df['energy(kWh/hh)'].notna().sum()

budgets = {
    'equipment_fault': 0.011 * n_valid,
    'degraded_sensor': 0.0165 * n_valid,
    'biased_reading': 0.0165 * n_valid,
    'corrupted_falsified': 0.0055 * n_valid,
    'natural_failure': 0.0055 * n_valid,
}
for k, v in budgets.items():
    print(f"{k} : {int(v)} lignes cibles")

positions_par_foyer = {f: df.index[df['LCLid'] == f].to_numpy() for f in foyers}
disponible = df['energy(kWh/hh)'].notna().to_numpy().copy()

df['anomaly_id'] = np.nan
df['anomaly_type'] = 'none'
df['is_data_quality_issue'] = False

equipment_fault : 10463 lignes cibles
degraded_sensor : 15695 lignes cibles
biased_reading : 15695 lignes cibles
corrupted_falsified : 5231 lignes cibles
natural_failure : 5231 lignes cibles


Fonctions d'effet par type d'anomalie

In [ ]:
def effet_equipment_fault(df, segment, rng):
    valeur_gel = df.loc[segment[0], 'energy(kWh/hh)']
    if pd.isna(valeur_gel):
        valeur_gel = 0.0
    df.loc[segment, 'energy(kWh/hh)'] = valeur_gel
    df.loc[segment, 'power_kW'] = valeur_gel * 2

def effet_degraded_sensor(df, segment, rng):
    n = len(segment)
    taux_derive = rng.uniform(0.10, 0.20)
    v0 = df.loc[segment, 'Voltage'].to_numpy()
    t = np.arange(n)
    derive = v0[0] * taux_derive * (t / n)
    bruit_croissant = rng.normal(0, 1, n) * (1 + 3 * t / n)
    df.loc[segment, 'Voltage'] = (v0 + derive + bruit_croissant).clip(190, 280)

def effet_biased_reading(df, segment, rng):
    facteur = rng.uniform(1.08, 1.20) if rng.random() > 0.5 else rng.uniform(0.80, 0.92)
    df.loc[segment, 'energy(kWh/hh)'] = df.loc[segment, 'energy(kWh/hh)'] * facteur
    df.loc[segment, 'power_kW'] = df.loc[segment, 'energy(kWh/hh)'] * 2

def effet_corrupted_falsified(df, segment, rng, positions_par_foyer):
    idx = segment[0]
    variante = rng.choice(['spike', 'negatif', 'rejeu'])
    if variante == 'spike':
        facteur = rng.uniform(3, 8)
        df.loc[idx, 'energy(kWh/hh)'] = df.loc[idx, 'energy(kWh/hh)'] * facteur
    elif variante == 'negatif':
        df.loc[idx, 'energy(kWh/hh)'] = -abs(df.loc[idx, 'energy(kWh/hh)'])
    else:  # rejeu
        foyer = df.loc[idx, 'LCLid']
        positions_foyer = positions_par_foyer[foyer]
        pos_dans_segment = np.searchsorted(positions_foyer, idx)
        if pos_dans_segment >= 48:
            valeur_passee = df.loc[positions_foyer[pos_dans_segment - 48], 'energy(kWh/hh)']
            if pd.notna(valeur_passee):
                df.loc[idx, 'energy(kWh/hh)'] = valeur_passee
    df.loc[idx, 'power_kW'] = df.loc[idx, 'energy(kWh/hh)'] * 2

def effet_natural_failure(df, segment, rng):
    effet_equipment_fault(df, segment, rng)

Fonction générique d'injection par épisodes

In [ ]:
def trouver_segment_libre(positions_foyer, disponible, duree, rng, max_essais=50):
    n = len(positions_foyer)
    if n <= duree:
        return None
    for _ in range(max_essais):
        start_local = rng.integers(0, n - duree)
        segment = positions_foyer[start_local:start_local + duree]
        if disponible[segment].all():
            return segment
    return None

def injecter_type(df, disponible, positions_par_foyer, poids_par_foyer, foyers,
                   type_nom, duree_min, duree_max, budget_lignes, effet_fn, rng,
                   anomaly_counter_start, filtre_meteo=None):
    lignes_injectees = 0
    anomaly_counter = anomaly_counter_start
    probs = np.array([poids_par_foyer[f] for f in foyers])
    probs = probs / probs.sum()

    tentatives = 0
    max_tentatives = max(int(budget_lignes / duree_min) * 8, 2000)

    while lignes_injectees < budget_lignes and tentatives < max_tentatives:
        tentatives += 1
        foyer = rng.choice(foyers, p=probs)
        positions_foyer = positions_par_foyer[foyer]
        duree = int(rng.integers(duree_min, duree_max + 1))

        if filtre_meteo is not None:
            positions_valides_meteo = positions_foyer[filtre_meteo[positions_foyer]]
            if len(positions_valides_meteo) < duree:
                continue
            idx_dep = rng.integers(0, len(positions_valides_meteo) - duree + 1)
            segment = positions_valides_meteo[idx_dep:idx_dep + duree]
            if not disponible[segment].all():
                continue
        else:
            segment = trouver_segment_libre(positions_foyer, disponible, duree, rng)
            if segment is None:
                continue

        if type_nom == 'corrupted_falsified':
            effet_fn(df, segment, rng, positions_par_foyer)
        else:
            effet_fn(df, segment, rng)

        df.loc[segment, 'anomaly_id'] = anomaly_counter
        df.loc[segment, 'anomaly_type'] = type_nom
        df.loc[segment, 'is_data_quality_issue'] = True
        disponible[segment] = False

        anomaly_counter += 1
        lignes_injectees += len(segment)

    print(f"{type_nom} : {lignes_injectees} lignes injectées ({tentatives} tentatives), "
          f"{anomaly_counter - anomaly_counter_start} épisodes")
    return anomaly_counter

Exécution des 5 injections dans l'ordre

In [ ]:
anomaly_counter = 0

anomaly_counter = injecter_type(df, disponible, positions_par_foyer, poids_par_foyer, foyers,
                                 'equipment_fault', 4, 48, budgets['equipment_fault'],
                                 effet_equipment_fault, rng, anomaly_counter)

anomaly_counter = injecter_type(df, disponible, positions_par_foyer, poids_par_foyer, foyers,
                                 'degraded_sensor', 96, 336, budgets['degraded_sensor'],
                                 effet_degraded_sensor, rng, anomaly_counter)

anomaly_counter = injecter_type(df, disponible, positions_par_foyer, poids_par_foyer, foyers,
                                 'biased_reading', 168, 720, budgets['biased_reading'],
                                 effet_biased_reading, rng, anomaly_counter)

anomaly_counter = injecter_type(df, disponible, positions_par_foyer, poids_par_foyer, foyers,
                                 'corrupted_falsified', 1, 3, budgets['corrupted_falsified'],
                                 effet_corrupted_falsified, rng, anomaly_counter)

equipment_fault : 10495 lignes injectées (400 tentatives), 400 épisodes
degraded_sensor : 15826 lignes injectées (77 tentatives), 77 épisodes
biased_reading : 15766 lignes injectées (35 tentatives), 35 épisodes
corrupted_falsified : 5234 lignes injectées (2608 tentatives), 2608 épisodes


In [ ]:
def calculer_runs_continus(mask_bool):
    diff = np.diff(np.concatenate(([0], mask_bool.astype(int), [0])))
    debuts = np.where(diff == 1)[0]
    fins = np.where(diff == -1)[0]
    return list(zip(debuts, fins))

timestamps_uniques = np.sort(df['tstp'].unique())

# Approximation du patron climatique "propre" : moyenne des 41 foyers à chaque instant
# (le bruit individuel de chaque foyer s'annule très largement en moyennant sur 41 tirages indépendants)
temp_moyennee = df.groupby('tstp')['Temperature_synth'].mean().reindex(timestamps_uniques)

SEUIL_HAUT, SEUIL_BAS = 32, 5
mask_extreme = (temp_moyennee > SEUIL_HAUT) | (temp_moyennee < SEUIL_BAS)

runs = calculer_runs_continus(mask_extreme.to_numpy())
runs = [(d, f) for d, f in runs if f - d >= 2]
print("Runs météo extrême exploitables :", len(runs))
print("Longueurs (créneaux) :", sorted([f-d for d,f in runs], reverse=True))
print("Total de créneaux extrêmes :", sum(f-d for d,f in runs))

Runs météo extrême exploitables : 28
Longueurs (créneaux) : [np.int64(24), np.int64(20), np.int64(20), np.int64(19), np.int64(18), np.int64(18), np.int64(18), np.int64(16), np.int64(14), np.int64(14), np.int64(14), np.int64(14), np.int64(10), np.int64(10), np.int64(10), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(6), np.int64(6), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(2), np.int64(2), np.int64(2)]
Total de créneaux extrêmes : 305


In [ ]:
def injecter_natural_failure(df, disponible, positions_par_foyer, poids_par_foyer, foyers,
                              runs, budget_lignes, rng, anomaly_counter_start):
    anomaly_counter = anomaly_counter_start
    lignes_injectees = 0
    probs = np.array([poids_par_foyer[f] for f in foyers])
    probs = probs / probs.sum()

    tentatives, max_tentatives = 0, 20000
    while lignes_injectees < budget_lignes and tentatives < max_tentatives and len(runs) > 0:
        tentatives += 1
        d, f = runs[rng.integers(0, len(runs))]
        longueur_run = f - d
        duree = min(int(rng.integers(2, 13)), longueur_run)
        offset = rng.integers(0, longueur_run - duree + 1) if longueur_run > duree else 0

        foyer = rng.choice(foyers, p=probs)
        segment = positions_par_foyer[foyer][d + offset : d + offset + duree]

        if not disponible[segment].all():
            continue

        effet_equipment_fault(df, segment, rng)
        df.loc[segment, 'anomaly_id'] = anomaly_counter
        df.loc[segment, 'anomaly_type'] = 'natural_failure'
        df.loc[segment, 'is_data_quality_issue'] = True
        disponible[segment] = False

        anomaly_counter += 1
        lignes_injectees += len(segment)

    print(f"natural_failure : {lignes_injectees} lignes injectées ({tentatives} tentatives), "
          f"{anomaly_counter - anomaly_counter_start} épisodes "
          f"(budget théorique max ≈ {len(foyers) * sum(f-d for d,f in runs)} lignes)")
    return anomaly_counter

anomaly_counter = injecter_natural_failure(df, disponible, positions_par_foyer, poids_par_foyer, foyers,
                                            runs, budgets['natural_failure'], rng, anomaly_counter)

natural_failure : 5233 lignes injectées (1970 tentatives), 905 épisodes (budget théorique max ≈ 12505 lignes)


Injection des événements réseau réels (genuine events)

In [ ]:
def injecter_genuine_events(df, n_episodes, duree_min, duree_max, foyers, rng, anomaly_counter_start):
    anomaly_counter = anomaly_counter_start
    timestamps_uniques = np.sort(df['tstp'].unique())

    for _ in range(n_episodes):
        duree = int(rng.integers(duree_min, duree_max + 1))
        idx_debut = rng.integers(0, len(timestamps_uniques) - duree)
        fenetre_temps = timestamps_uniques[idx_debut:idx_debut + duree]

        fraction_foyers = rng.uniform(0.7, 1.0)
        foyers_touches = rng.choice(foyers, size=int(fraction_foyers * len(foyers)), replace=False)

        sens = rng.choice(['hausse', 'baisse'])
        amplitude_base = rng.uniform(1.5, 3.0) if sens == 'hausse' else rng.uniform(0.2, 0.5)

        mask = df['tstp'].isin(fenetre_temps) & df['LCLid'].isin(foyers_touches)
        n_lignes = mask.sum()
        bruit_individuel = rng.normal(1.0, 0.1, n_lignes)

        df.loc[mask, 'energy(kWh/hh)'] = df.loc[mask, 'energy(kWh/hh)'] * amplitude_base * bruit_individuel
        df.loc[mask, 'power_kW'] = df.loc[mask, 'energy(kWh/hh)'] * 2
        df.loc[mask, 'anomaly_id'] = anomaly_counter
        df.loc[mask, 'anomaly_type'] = 'genuine_event'

        anomaly_counter += 1

    print(f"genuine_event : {anomaly_counter - anomaly_counter_start} épisodes injectés")
    return anomaly_counter

anomaly_counter = injecter_genuine_events(df, 12, 2, 12, foyers, rng, anomaly_counter)

genuine_event : 12 épisodes injectés


Feature de corrélation spatiale (z-score personnel + fraction de foyers extrêmes)

In [ ]:
# Bloc 1 — Baseline calculée sur le dataset AVANT injection (jamais sur les données contaminées)
df_reference = pd.read_csv(base + 'quartier_unifie.csv', parse_dates=['tstp'])
df_reference['heure'] = df_reference['tstp'].dt.hour
df_reference['jour_semaine'] = df_reference['tstp'].dt.dayofweek

baseline_propre = df_reference.groupby(['LCLid', 'heure', 'jour_semaine'])['energy(kWh/hh)'].agg(
    mediane='median',
    mad=lambda x: (x - x.median()).abs().median()
).reset_index()

df_reference = df_reference.merge(baseline_propre, on=['LCLid', 'heure', 'jour_semaine'], how='left')
df_reference['z_score_personnel'] = (
    (df_reference['energy(kWh/hh)'] - df_reference['mediane']) / (1.4826 * df_reference['mad'].replace(0, np.nan))
)

print(df_reference['z_score_personnel'].describe())

count    951249.000000
mean          2.944189
std          25.562130
min          -7.507375
25%          -0.592458
50%           0.000000
75%           0.880842
max         454.381942
Name: z_score_personnel, dtype: float64


In [ ]:
# Bloc 2 — Calibration du seuil PAR FOYER (99e percentile des |z| sur données saines)
seuils_calibres = df_reference.groupby('LCLid')['z_score_personnel'].apply(
    lambda x: x.abs().quantile(0.99)
).rename('seuil_z_calibre').reset_index()

print(seuils_calibres['seuil_z_calibre'].describe())

count     41.000000
mean      18.579246
std       52.485569
min        3.194862
25%        7.653257
50%        9.149815
75%       13.517963
max      345.001990
Name: seuil_z_calibre, dtype: float64


In [ ]:
# Bloc 3 — Plafonnement pour neutraliser le foyer atypique (MAC003422 et voisins)
plafond = seuils_calibres['seuil_z_calibre'].quantile(0.90)
print("Plafond appliqué :", plafond)

seuils_calibres['seuil_z_calibre'] = seuils_calibres['seuil_z_calibre'].clip(upper=plafond)
print(seuils_calibres.sort_values('seuil_z_calibre', ascending=False).head(5))

Plafond appliqué : 15.468321417329914
        LCLid  seuil_z_calibre
8   MAC003394        15.468321
5   MAC003281        15.468321
12  MAC003428        15.468321
11  MAC003423        15.468321
10  MAC003422        15.468321


In [ ]:
# Bloc 4 — Application des seuils calibrés/plafonnés sur df (dataset avec anomalies)
df['heure'] = df['tstp'].dt.hour
df['jour_semaine'] = df['tstp'].dt.dayofweek
df = df.merge(baseline_propre, on=['LCLid', 'heure', 'jour_semaine'], how='left')

df['z_score_personnel'] = (
    (df['energy(kWh/hh)'] - df['mediane']) / (1.4826 * df['mad'].replace(0, np.nan))
)

df = df.drop(columns=['seuil_z_calibre'], errors='ignore').merge(
    seuils_calibres, on='LCLid', how='left')

df['est_extreme'] = df['z_score_personnel'].abs() > df['seuil_z_calibre']

In [ ]:
# Bloc 5 — Recalcul de la fraction de foyers extrêmes par instant, et vérification finale
fraction_extreme_par_temps = df.groupby('tstp')['est_extreme'].mean().rename('fraction_foyers_extremes')
df = df.drop(columns=['fraction_foyers_extremes'], errors='ignore').merge(
    fraction_extreme_par_temps, on='tstp', how='left')

print(df.groupby('anomaly_type')['fraction_foyers_extremes'].mean().sort_values(ascending=False))

anomaly_type
genuine_event          0.077895
corrupted_falsified    0.025148
equipment_fault        0.023606
none                   0.021677
degraded_sensor        0.019890
biased_reading         0.019345
natural_failure        0.008902
Name: fraction_foyers_extremes, dtype: float64


Vérifications finales et sauvegarde

In [ ]:
print(df['anomaly_type'].value_counts())
print("\nTaux global d'anomalies data-quality :", df['is_data_quality_issue'].mean() * 100, "%")

df.to_csv('/content/quartier_avec_anomalies.csv', index=False)
df.to_csv(base + 'quartier_avec_anomalies.csv', index=False)
print(df.shape)

anomaly_type
none                   897013
degraded_sensor         15759
biased_reading          15722
equipment_fault         10447
natural_failure          5233
corrupted_falsified      5214
genuine_event            3165
Name: count, dtype: int64

Taux global d'anomalies data-quality : 5.517173322639265 %
(952553, 24)
